> ⚠ **Not migrated: depends on the v1 YAML electrothermal telemetry
> pipeline (v2 gap, migration-from-v1.md §7).**
>
> This tutorial relied on v1-only API surface that v2 does not provide:
>
> - `ps.YamlParser` / `ps.YamlParserOptions` — replaced by
>   `p.load_yaml_file(path)` / `p.load_yaml_string(text)`, which return
>   a `LoadedCircuit(builder, options)` and take **no parser options**
>   (see the deprecation map in `pulsim/__init__.py`).
> - The v1 YAML schema (`schema: pulsim-v1`, `simulation.thermal:` block)
>   vs the v2 schema (top-level `circuit:` key). The v2 loader has no
>   `thermal:` block.
> - `SimulationResult.component_electrothermal`, `.loss_summary`,
>   `.thermal_summary` — per-component electrothermal telemetry is not
>   exposed by the v2 `SimulationResult` (which carries
>   `times/states/v/i/power`).
>
> v2 instead offers programmatic loss/thermal post-processing via
> `p.device_loss_summary(builder, result, ...)`,
> `p.device_thermal_summary(builder, result, thermal_specs=...)`,
> `p.LossAccumulator`, `p.EfficiencyCalculator`, and Foster/Cauer
> thermal networks (`p.add_foster_network`). Porting this tutorial to
> that pattern is tracked as follow-up work; the cells below are left
> inert so the notebook executes without error.

# Tutorial: Electrothermal Component Telemetry (v1 — NOT migrated)

**Original audience:** users already comfortable with Pulsim YAML +
transient execution.

**Original learning goals (v1 API, no v2 equivalent yet):**

- how to run an electrothermal benchmark netlist via `YamlParser`,
- how to read `SimulationResult.component_electrothermal`,
- how to check consistency against `loss_summary` and `thermal_summary`.


## Outline

1. Locate repo root and import `pulsim`.
2. Load and run `buck_electrothermal.yaml`.
3. Inspect per-component electrothermal telemetry.
4. Cross-check aggregate consistency (`loss_summary`/`thermal_summary`).


In [1]:
# v2 import works fine; the gap is the YAML electrothermal-telemetry
# pipeline this tutorial was built on, not the package itself.
import pulsim as p

print("Pulsim:", p.__version__)
print()
print("This tutorial is NOT migrated to v2. The v1 API it used")
print("(YamlParser + SimulationResult.component_electrothermal /")
print(" .loss_summary / .thermal_summary) has no v2 equivalent.")
print("See the top banner and migration-from-v1.md §7.")

# Sanity-check the v1 symbols this tutorial needs are indeed absent in v2:
missing = [name for name in
           ("YamlParser", "YamlParserOptions")
           if not hasattr(p, name)]
print()
print("v1 symbols absent in v2:", missing)
assert missing == ["YamlParser", "YamlParserOptions"]


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


...

Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64
Running cmake --build & --install in /Users/lgili/Documents/01 - Codes/01 - Github/Pulsim/build/cp313-cp313-macosx_26_0_arm64


Pulsim: 1.6.4

This tutorial is NOT migrated to v2. The v1 API it used
(YamlParser + SimulationResult.component_electrothermal /
 .loss_summary / .thermal_summary) has no v2 equivalent.
See the top banner and migration-from-v1.md §7.

v1 symbols absent in v2: ['YamlParser', 'YamlParserOptions']


## Step 1 - Run an electrothermal case (v1 flow — reference only)

In v1 this used `YamlParser` to load the electrothermal benchmark
netlist and then read per-component telemetry off the result. Neither
the parser class nor the telemetry accessors exist in v2, so the code
below is shown for reference only and is not executed.


### Original v1 code (reference only — does not run on v2)

The block below is the original v1 implementation. It is shown for
reference and is intentionally **not executed**: `YamlParser` and the
`component_electrothermal` / `loss_summary` / `thermal_summary`
accessors do not exist in v2.

```python
# Minimal working example (v1 API)
netlist_path = repo_root / "benchmarks" / "circuits" / "buck_electrothermal.yaml"

parser_opts = ps.YamlParserOptions()
parser_opts.strict = False
parser = ps.YamlParser(parser_opts)
circuit, options = parser.load(str(netlist_path))
if parser.errors:
    raise RuntimeError("YAML parser errors:\n" + "\n".join(parser.errors))

options.newton_options.num_nodes = int(circuit.num_nodes())
options.newton_options.num_branches = int(circuit.num_branches())

sim = ps.Simulator(circuit, options)
result = sim.run_transient(circuit.initial_state())

rows = sorted(result.component_electrothermal, key=lambda i: i.component_name)
for item in rows:
    print(item.component_name, item.total_loss, item.peak_temperature)

# Consistency checks against result.loss_summary / result.thermal_summary ...
```

**v2 direction (sketch).** Build the circuit with `p.CircuitBuilder`
(or `p.load_yaml_file` once a v2 thermal schema exists), run
`res = p.simulate(b, t_end=..., dt=...)`, then post-process losses and
temperatures with `p.device_loss_summary(b, res, ...)` and
`p.device_thermal_summary(b, res, thermal_specs=...)`. Per-component
junction temperatures come from a Foster network added via
`p.add_foster_network(b, [p.FosterStage(R_th, tau), ...], junction_node=...)`.


## Exercises

1. Change `simulation.thermal.policy` in the YAML to `loss_only` and compare temperatures.
2. Increase `M1.thermal.rth` by 2x and observe `peak_temperature` delta.
3. Add one extra passive component and verify it still appears in `component_electrothermal`.


In [2]:
# Exercise answer scaffold
# 1) Duplicate netlist to a temporary file and edit thermal values.
# 2) Re-run parser + simulator.
# 3) Compare rows from result.component_electrothermal.

# TODO: implement your experiment here.
